# Physics-Based Validation Layer for Power Grid Operation (Pre-ML)

This notebook implements a physics-based validation layer for synchrophasor power grid
monitoring. The purpose of this layer is to evaluate whether observed grid behavior is
physically plausible based on electrical principles, independent of cyber intent,
attack labels, or operational context.

All runtime features are retained and categorized; however, only measurements with direct
physical interpretation are used to assess physical stress or violations. Absolute phase
angles, control statuses, and cyber-related features are excluded from physical severity
assessment, as they may vary during normal operating conditions such as load changes or
maintenance without indicating unsafe behavior.

Each grid element (generators, transmission lines, and protection relays) is assigned a
continuous health value derived from the maximum physical deviation across its associated
measurements. For interpretability, these health values may be visualized using a
green–amber–red color scheme:

• **GREEN** — Measurements are within normal physical operating limits.  
• **AMBER** — Measurements indicate elevated but physically plausible stress or transient
  deviation.  
• **RED** — Measurements exhibit abnormal physical behavior outside expected operating
  limits.

A red state reflects a physical anomaly only and does not imply malicious intent or attack
activity.

This layer does not perform attack detection or classification. Instead, it provides a
physics-consistent foundation for subsequent machine learning, interpretation, and
operational decision-support layers.


In [74]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "../data/merged/multi_class_dataset_clean_FULL.csv"
df = pd.read_csv(DATA_PATH)

assert "marker" in df.columns, "Need marker column in training/offline dataset"
print(df.shape)
print("markers:", sorted(df["marker"].unique())[:10], "...")

(78377, 133)
markers: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] ...


In [77]:
RELAYS = ["R1", "R2", "R3", "R4"]

def is_physical_col(c: str) -> bool:
    c = str(c).strip()  # safety against hidden spaces
    
    for r in RELAYS:
        # Pattern A: R1-...
        if c.startswith(r + "-"):
            return True
        
        # Pattern B: R1:...
        if c.startswith(r + ":"):
            return True
        
        # Explicit impedance flag safeguard
        if c.startswith(r + "-PA:Z_inf_flag"):
            return True

    return False


PHYSICAL_FEATURES = [c for c in df.columns if is_physical_col(c)]

print("Physical features found:", len(PHYSICAL_FEATURES))
print("Flag columns found:", [c for c in PHYSICAL_FEATURES if "Z_inf_flag" in c])
print("Example:", PHYSICAL_FEATURES[:120])

Physical features found: 120
Flag columns found: ['R1-PA:Z_inf_flag', 'R2-PA:Z_inf_flag', 'R3-PA:Z_inf_flag', 'R4-PA:Z_inf_flag']
Example: ['R1-PA1:VH', 'R1-PM1:V', 'R1-PA2:VH', 'R1-PM2:V', 'R1-PA3:VH', 'R1-PM3:V', 'R1-PA4:IH', 'R1-PM4:I', 'R1-PA5:IH', 'R1-PM5:I', 'R1-PA6:IH', 'R1-PM6:I', 'R1-PA7:VH', 'R1-PM7:V', 'R1-PA8:VH', 'R1-PM8:V', 'R1-PA9:VH', 'R1-PM9:V', 'R1-PA10:IH', 'R1-PM10:I', 'R1-PA11:IH', 'R1-PM11:I', 'R1-PA12:IH', 'R1-PM12:I', 'R1:F', 'R1:DF', 'R1-PA:Z', 'R1-PA:ZH', 'R1:S', 'R2-PA1:VH', 'R2-PM1:V', 'R2-PA2:VH', 'R2-PM2:V', 'R2-PA3:VH', 'R2-PM3:V', 'R2-PA4:IH', 'R2-PM4:I', 'R2-PA5:IH', 'R2-PM5:I', 'R2-PA6:IH', 'R2-PM6:I', 'R2-PA7:VH', 'R2-PM7:V', 'R2-PA8:VH', 'R2-PM8:V', 'R2-PA9:VH', 'R2-PM9:V', 'R2-PA10:IH', 'R2-PM10:I', 'R2-PA11:IH', 'R2-PM11:I', 'R2-PA12:IH', 'R2-PM12:I', 'R2:F', 'R2:DF', 'R2-PA:Z', 'R2-PA:ZH', 'R2:S', 'R3-PA1:VH', 'R3-PM1:V', 'R3-PA2:VH', 'R3-PM2:V', 'R3-PA3:VH', 'R3-PM3:V', 'R3-PA4:IH', 'R3-PM4:I', 'R3-PA5:IH', 'R3-PM5:I', 'R3-PA6:IH', 'R3-PM6:I', 'R

In [82]:
ANGLE_TYPES = {"voltage_phase_angle", "current_phase_angle", "impedance_angle"}

def feature_type(col: str) -> str:
    col = str(col).strip()

    # ---- Impedance flag FIRST ----
    if "Z_inf_flag" in col:
        return "impedance_flag"

    # ---- Status ----
    if col.endswith(":S"):
        return "status_flag"

    # ---- Frequency ----
    if col.endswith(":F"):
        return "frequency"

    if col.endswith(":DF"):
        return "rocof"

    # ---- Angles ----
    if col.endswith(":VH"):
        return "voltage_phase_angle"

    if col.endswith(":IH"):
        return "current_phase_angle"

    if col.endswith(":ZH"):
        return "impedance_angle"

    # ---- Magnitudes ----
    if col.endswith(":V"):
        return "voltage_magnitude"

    if col.endswith(":I"):
        return "current_magnitude"

    if col.endswith(":Z"):
        return "impedance_magnitude"

    return "other"
    return "other"
    
FEATURE_TYPES = {c: feature_type(c) for c in PHYSICAL_FEATURES}
print("features:", len(FEATURE_TYPES))
# quick sanity counts
from collections import Counter
print(Counter(FEATURE_TYPES.values()))

features: 120
Counter({'voltage_phase_angle': 24, 'voltage_magnitude': 24, 'current_phase_angle': 24, 'current_magnitude': 24, 'frequency': 4, 'rocof': 4, 'impedance_magnitude': 4, 'impedance_angle': 4, 'status_flag': 4, 'impedance_flag': 4})


In [84]:
STRESS_TYPES = {
    "voltage_magnitude",
    "current_magnitude",
    "impedance_magnitude",
    "frequency",
    "rocof"
}

STRESS_FEATURES = [
    c for c in PHYSICAL_FEATURES
    if FEATURE_TYPES.get(c) in STRESS_TYPES
]

print("Stress features:", len(STRESS_FEATURES))
from collections import Counter

Counter([FEATURE_TYPES[c] for c in STRESS_FEATURES])

Stress features: 60


Counter({'voltage_magnitude': 24,
         'current_magnitude': 24,
         'frequency': 4,
         'rocof': 4,
         'impedance_magnitude': 4})

In [85]:
# -----------------------------------------
# Build baseline from normal operation
# -----------------------------------------

normal_mask = df["marker"] == 41   # your normal class

RELAYS = ["R1", "R2", "R3", "R4"]

# Group stress features by relay
RELAY_STRESS_FEATURES = {
    r: [c for c in STRESS_FEATURES if c.startswith(r)]
    for r in RELAYS
}

# Compute baseline mean + std per relay
BASELINE_MEAN = {}
BASELINE_STD = {}

for r, cols in RELAY_STRESS_FEATURES.items():
    BASELINE_MEAN[r] = df.loc[normal_mask, cols].mean()
    BASELINE_STD[r] = df.loc[normal_mask, cols].std()

print("Baseline built.")

Baseline built.


In [35]:
# -------------------------------------------------
# Build baseline from normal operation
# -------------------------------------------------

normal_mask = df["marker"] == 41

BASELINE_MEAN = {}
BASELINE_STD  = {}

for r, cols in RELAY_FEATURES.items():
    BASELINE_MEAN[r] = df.loc[normal_mask, cols].mean()
    BASELINE_STD[r]  = df.loc[normal_mask, cols].std()

In [42]:
import numpy as np

def relay_abnormality(row, relay):
    cols = RELAY_FEATURES[relay]
    
    z = (row[cols] - BASELINE_MEAN[relay]) / (BASELINE_STD[relay] + 1e-6)
    
    return np.abs(z).mean()

In [43]:
relay_scores = []

for _, row in df.iterrows():
    scores = {}
    for r in RELAYS:
        scores[r] = relay_abnormality(row, r)
    relay_scores.append(scores)

relay_df = pd.DataFrame(relay_scores, index=df.index)

print("Relay abnormality shape:", relay_df.shape)
relay_df.head()

Relay abnormality shape: (78377, 4)


,R1,R2,R3,R4
0,1.587423,1.393546,1.406965,1.606381
1,1.704129,1.882904,1.849510,1.644675
2,1.600732,1.765914,1.706260,1.551007
3,0.764743,0.875151,0.876140,0.765285
4,0.751121,0.840384,0.843800,0.752179


In [44]:
global_max = relay_df.max().max()
relay_df = relay_df / (global_max + 1e-6)

In [45]:
relay_df["L1"] = relay_df[["R1","R2"]].max(axis=1)
relay_df["L2"] = relay_df[["R3","R4"]].max(axis=1)

relay_df.head()

,R1,R2,R3,R4,L1,L2
0,0.009551,0.008384,0.008465,0.009665,0.009551,0.009665
1,0.010253,0.011329,0.011128,0.009895,0.011329,0.011128
2,0.009631,0.010625,0.010266,0.009332,0.010625,0.010266
3,0.004601,0.005265,0.005271,0.004604,0.005265,0.005271
4,0.004519,0.005056,0.005077,0.004526,0.005056,0.005077


In [40]:
def to_colour(x):
    if x < 0.25: return 0.0
    if x < 0.5:  return 0.3
    if x < 0.75: return 0.6
    return 1.0

colour_df = relay_df.applymap(to_colour)

/var/folders/nv/f9_p0fkx4xsf4ygmqb8ynm9w0000gn/T/ipykernel_54897/664066478.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  colour_df = relay_df.applymap(to_colour)


In [3]:
# --------------------------------------------------
# Relay / Breaker Status Sanity Check
# --------------------------------------------------

print("=== Relay Status Sanity Check ===")

status_cols = [
    c for c in FEATURES_ALL
    if c.endswith(":S") and c.startswith("R")
]

if not status_cols:
    raise ValueError("❌ No relay status columns found (R*:S)")

print(f"Found {len(status_cols)} relay status columns:")
print(status_cols)
print()

# Check unique values
for col in status_cols:
    vals = sorted(df[col].dropna().unique())
    print(f"{col}: {vals}")

# Check variability across dataset
print("\n=== Status Change Counts ===")
for col in status_cols:
    changes = (df[col].diff() != 0).sum()
    print(f"{col}: {changes} changes")

=== Relay Status Sanity Check ===
Found 4 relay status columns:
['R1:S', 'R2:S', 'R3:S', 'R4:S']

R1:S: [np.int64(0)]
R2:S: [np.int64(0)]
R3:S: [np.int64(0)]
R4:S: [np.int64(0)]

=== Status Change Counts ===
R1:S: 1 changes
R2:S: 1 changes
R3:S: 1 changes
R4:S: 1 changes


In [27]:
# --------------------------------------------------
# Quiet baseline selection and feature statistics
# --------------------------------------------------

print("=== Quiet Baseline Selection ===")

# Define physically quiet operating condition
quiet_mask = (
    (df["marker"] == 41) &
    (df["R1:DF"].abs() < 0.05) &
    (df["R2:DF"].abs() < 0.05) &
    (df["R3:DF"].abs() < 0.05) &
    (df["R4:DF"].abs() < 0.05)
)

# Extract quiet baseline data
df_quiet = df.loc[quiet_mask, FEATURES_ALL]

print(f"Total samples        : {len(df)}")
print(f"Quiet baseline samples: {len(df_quiet)}")
print(f"Quiet ratio           : {len(df_quiet) / len(df):.2%}")
print()

print("=== Computing Feature Statistics ===")

# Compute robust feature statistics
stats = []
skipped_few_samples = 0

for col in FEATURES_ALL:

    # Safety check
    if col not in FEATURE_TYPES:
        continue

    s = df_quiet[col].dropna()

    # Require sufficient samples for stability
    if len(s) < 50:
        skipped_few_samples += 1
        continue

    stats.append({
        "feature": col,
        "type": FEATURE_TYPES[col],
        "median": float(s.median()),
        "q25": float(s.quantile(0.25)),
        "q75": float(s.quantile(0.75)),
        "p90": float(s.quantile(0.90)),
        "p95": float(s.quantile(0.95)),
        "p97": float(s.quantile(0.97)),
        "p99": float(s.quantile(0.99)),
    })

# Build statistics DataFrame
feature_stats_df = pd.DataFrame(stats).set_index("feature")

print(f"Features with statistics : {len(feature_stats_df)}")
print(f"Features skipped (<50 samples): {skipped_few_samples}")
print()

print("=== Feature Type Breakdown ===")
print(feature_stats_df["type"].value_counts())
print()

# Inspect head
feature_stats_df.head()

# Save calibration statistics
feature_stats_df.to_json(
    "FEATURE_STATS.json",
    orient="index",
    indent=2
)

print("Saved FEATURE_STATS.json")


=== Quiet Baseline Selection ===
Total samples        : 78377
Quiet baseline samples: 4405
Quiet ratio           : 5.62%

=== Computing Feature Statistics ===
Features with statistics : 112
Features skipped (<50 samples): 0

=== Feature Type Breakdown ===
type
voltage_phase_angle    24
voltage_magnitude      24
current_phase_angle    24
current_magnitude      24
frequency               4
rocof                   4
impedance_flag          4
impedance_angle         4
Name: count, dtype: int64

Saved FEATURE_STATS.json


In [28]:
fault_mask = df["marker"] == 1
normal_mask = df["marker"] == 41

print("Fault rows:", fault_mask.sum())
print("Normal rows:", normal_mask.sum())

Fault rows: 1643
Normal rows: 4405


In [30]:
# get current magnitude columns
current_cols = [c for c in PHYSICAL_FEATURES if FEATURE_TYPES[c] == "current_magnitude"]

# split by relay
R1_cols = [c for c in current_cols if c.startswith("R1-")]
R2_cols = [c for c in current_cols if c.startswith("R2-")]
R3_cols = [c for c in current_cols if c.startswith("R3-")]
R4_cols = [c for c in current_cols if c.startswith("R4-")]

def relay_mean_current(cols, mask):
    return df.loc[mask, cols].mean().mean()

print("NORMAL")
print("R1:", relay_mean_current(R1_cols, normal_mask))
print("R2:", relay_mean_current(R2_cols, normal_mask))
print("R3:", relay_mean_current(R3_cols, normal_mask))
print("R4:", relay_mean_current(R4_cols, normal_mask))

print("\nFAULT L1")
print("R1:", relay_mean_current(R1_cols, fault_mask))
print("R2:", relay_mean_current(R2_cols, fault_mask))
print("R3:", relay_mean_current(R3_cols, fault_mask))
print("R4:", relay_mean_current(R4_cols, fault_mask))

NORMAL
R1: 261.2457701401816
R2: 264.61306594298145
R3: 262.5908561556943
R4: 259.3073384990541

FAULT L1
R1: 224.6316691935484
R2: 230.2813022910327
R3: 288.4426715424021
R4: 284.5704590390039


In [32]:
print("Impedance columns:")
print([c for c in PHYSICAL_FEATURES if "Z" in c])

Impedance columns:
['R1-PA:Z', 'R1-PA:ZH', 'R2-PA:Z', 'R2-PA:ZH', 'R3-PA:Z', 'R3-PA:ZH', 'R4-PA:Z', 'R4-PA:ZH']


In [33]:
print(df[[c for c in PHYSICAL_FEATURES if "Z" in c]].describe())

            R1-PA:Z      R1-PA:ZH       R2-PA:Z      R2-PA:ZH       R3-PA:Z  \
count  78377.000000  78377.000000  78377.000000  78377.000000  78377.000000   
mean      10.954697      0.012048     10.745735     -3.026418     10.887280   
std       12.562514      0.078730     13.379261      0.077862     14.462044   
min        0.185210     -0.161385      0.000000     -3.141593      0.000000   
25%        8.291006     -0.028589      8.051251     -3.079511      8.100873   
50%        9.957634      0.016968      9.705397     -3.049320      9.768775   
75%       12.249099      0.059942     11.968355     -3.000335     12.036460   
max     1306.246561      0.192739   2291.464914     -2.881571   1629.988981   

           R3-PA:ZH       R4-PA:Z      R4-PA:ZH  
count  78377.000000  78377.000000  78377.000000  
mean      -3.028907     11.280438      0.010632  
std        0.074375     41.197034      0.076807  
min       -3.141593      0.006781     -0.157996  
25%       -3.079122      8.347653     

In [11]:
def get_physical_bounds(row):
    t = row["type"]
    name = str(row.name)

    # --------------------------------------------------
    # Structural zeros (no physical sensor)
    # --------------------------------------------------
    if t == "voltage_magnitude" and any(k in name for k in ["PM8", "PM9"]):
        return {
            "normal": (0.0, 0.0),
            "warning": None,
            "critical": None
        }

    # --------------------------------------------------
    # Voltage magnitude (per-unit, grid code aligned)
    # --------------------------------------------------
    if t == "voltage_magnitude":
        mu = float(row["median"])
        return {
            "normal": (0.95 * mu, 1.05 * mu),
            "warning": (0.90 * mu, 1.10 * mu),
            "critical": (0.85 * mu, 1.15 * mu),
        }

    # --------------------------------------------------
    # Current magnitude (thermal stress model)
    # --------------------------------------------------
    if t == "current_magnitude":
        i90 = max(float(row["p90"]), 1e-6)
        i97 = max(float(row["p97"]), i90 * 1.05)
    
        return {
            "normal":   (0.0, i90),
            "warning":  (i90, i97),
            "critical": (i97, np.inf),
        }
    # --------------------------------------------------
    # Frequency (grid operational limits)
    # --------------------------------------------------
    if t == "frequency":
        return {
            "normal": (59.5, 60.5),
            "warning": (59.0, 61.0),
            "critical": (58.0, 62.0),
        }

    # --------------------------------------------------
    # ROCOF (system stability indicator)
    # --------------------------------------------------
    if t == "rocof":
        return {
            "normal": (-1.0, 1.0),
            "warning": (-2.0, 2.0),
            "critical": (-5.0, 5.0),
        }

    # --------------------------------------------------
    # Impedance infinite flag (PROTECTION DECISION)
    # --------------------------------------------------
    if t == "impedance_flag":
    # Binary physical protection output
        return {
            "normal": (0, 0),
            "warning": None,
            "critical": (1, 1),
        }

    # --------------------------------------------------
    # Angle-based & non-physical signals
    # --------------------------------------------------
    # impedance (raw Z)
    # voltage_phase_angle
    # current_phase_angle
    # impedance_angle
    # status bits, flags, logs
    return None

In [12]:
# --------------------------------------------------
# Physical Bounds
# Apply physical bounds, preview representative R1 bounds, and export
# --------------------------------------------------

print("=== Physical Bounds ===")

# Apply bounds to all features
feature_stats_df["bounds"] = feature_stats_df.apply(
    get_physical_bounds,
    axis=1
)

# Summary
total_features = len(feature_stats_df)
bounded_features = feature_stats_df["bounds"].notna().sum()
ignored_features = total_features - bounded_features

print(f"Total features with stats : {total_features}")
print(f"Physically bounded        : {bounded_features}")
print(f"Ignored (non-physical)    : {ignored_features}")
print()

# --------------------------------------------------
# Preview representative bounds for Relay R1
# --------------------------------------------------

print("=== R1 Physical Bounds (Preview) ===")

r1_bounds_df = feature_stats_df.loc[
    feature_stats_df.index.str.startswith("R1-") &
    feature_stats_df["bounds"].notna(),
    ["type", "bounds"]
].copy()

def unpack_bounds(b):
    return pd.Series({
        "normal": b.get("normal"),
        "warning": b.get("warning"),
        "critical": b.get("critical"),
    })

bounds_expanded = r1_bounds_df["bounds"].apply(unpack_bounds)

r1_bounds_table = pd.concat(
    [r1_bounds_df.drop(columns=["bounds"]), bounds_expanded],
    axis=1
)

print(f"Number of R1 physical features: {len(r1_bounds_table)}")
display(r1_bounds_table.sort_index())
print()

# --------------------------------------------------
# Export physical bounds (JSON-safe)
# --------------------------------------------------

physical_bounds_json = {}

for feature, row in feature_stats_df.iterrows():
    bounds = row["bounds"]

    if bounds is None:
        physical_bounds_json[feature] = {
            "type": row["type"],
            "bounds": None
        }
    else:
        physical_bounds_json[feature] = {
            "type": row["type"],
            "bounds": {
                k: list(v) if v is not None else None
                for k, v in bounds.items()
            }
        }

with open("PHYSICAL_FEATURE_BOUNDS.json", "w") as f:
    json.dump(physical_bounds_json, f, indent=2)

print("Saved PHYSICAL_FEATURE_BOUNDS.json")


=== Physical Bounds ===
Total features with stats : 132
Physically bounded        : 60
Ignored (non-physical)    : 72

=== R1 Physical Bounds (Preview) ===
Number of R1 physical features: 13


,type,normal,warning,critical
feature,,,,
R1-PA:Z_inf_flag,impedance_flag,"(0, 0)",None,"(1, 1)"
R1-PM10:I,current_magnitude,"(0.0, 501.17207)","(501.17207, 526.2306735000001)","(526.2306735000001, inf)"
R1-PM11:I,current_magnitude,"(0.0, 4.57775)","(4.57775, 13.16194680000002)","(13.16194680000002, inf)"
R1-PM12:I,current_magnitude,"(0.0, 4.94397)","(4.94397, 12.8177)","(12.8177, inf)"
R1-PM1:V,voltage_magnitude,"(125076.75375999999, 138242.72784)","(118493.76672, 144825.71488)","(111910.77967999999, 151408.70192)"
R1-PM2:V,voltage_magnitude,"(125029.11448999999, 138190.07391)","(118448.63478, 144770.55362)","(111868.15507, 151351.03332999998)"
R1-PM3:V,voltage_magnitude,"(125124.392935, 138295.38166500002)","(118538.89857, 144880.87603)","(111953.404205, 151466.37039499998)"
R1-PM4:I,current_magnitude,"(0.0, 500.98896)","(500.98896, 526.038408)","(526.038408, inf)"
R1-PM5:I,current_magnitude,"(0.0, 502.380596)","(502.380596, 527.4996258000001)","(527.4996258000001, inf)"



Saved PHYSICAL_FEATURE_BOUNDS.json


In [13]:
# --------------------------------------------------
# Feature-level physical severity computation (FINAL, CORRECT)
# --------------------------------------------------

print("=== Feature-Level Physical Severity ===")

def feature_severity(value, bounds):
    """
    Returns:
      0.0 = GREEN
      0.6 = AMBER
      1.0 = RED

    Supports:
      - Continuous physical signals (V, I, F, ROCOF, Z)
      - Discrete protection flags (impedance_flag)
    """
    if bounds is None or pd.isna(value):
        return 0.0

    # --------------------------------------------------
    # DISCRETE FLAG (impedance_flag)
    # --------------------------------------------------
    # Identified by having no warning band
    if bounds.get("warning") is None and bounds.get("critical") is not None:
        crit_lo, crit_hi = bounds["critical"]
        return 1.0 if crit_lo <= value <= crit_hi else 0.0

    # --------------------------------------------------
    # CONTINUOUS SIGNALS
    # --------------------------------------------------
    lo, hi = bounds["normal"]
    if lo <= value <= hi:
        return 0.0

    lo, hi = bounds["warning"]
    if lo <= value <= hi:
        return 0.6

    lo, hi = bounds["critical"]
    if value < lo or value > hi:
        return 1.0

    return 0.0


# --------------------------------------------------
# Build severity DataFrame
# --------------------------------------------------

severity_df = pd.DataFrame(
    {
        feature: df[feature].apply(
            lambda v: feature_severity(v, row["bounds"])
        )
        for feature, row in feature_stats_df.iterrows()
    },
    index=df.index
)

print(f"Severity matrix shape: {severity_df.shape}")
print("\nSeverity value counts:")
print(pd.Series(severity_df.values.ravel()).value_counts())

=== Feature-Level Physical Severity ===
Severity matrix shape: (78377, 132)

Severity value counts:
0.0    10085032
0.6      249826
1.0       10906
Name: count, dtype: int64


In [14]:
# --------------------------------------------------
# NORMAL operation validation (marker = 41)
# --------------------------------------------------

NORMAL_MARKER = 41
normal_idx = df[df["marker"] == NORMAL_MARKER].index

print("=== NORMAL OPERATION TEST ===")
print(f"Normal samples: {len(normal_idx)}")
print()

# -----------------------------
# Feature-level checks
# -----------------------------
critical_features = (severity_df.loc[normal_idx] == 1.0).sum().sum()
warning_features  = (severity_df.loc[normal_idx] == 0.6).sum().sum()

print("Feature-level severity:")
print(f"  CRITICAL count : {critical_features}")
print(f"  WARNING count  : {warning_features}")
print()

=== NORMAL OPERATION TEST ===
Normal samples: 4405

Feature-level severity:
  CRITICAL count : 0
  WARNING count  : 8396



In [15]:
import numpy as np
import pandas as pd

# ==================================================
# 1. Helper functions
# ==================================================

def current_cols(cols):
    """Current magnitude only (exclude angles)."""
    return [c for c in cols if FEATURE_TYPES.get(c) == "current_magnitude"]

def flag_cols(cols):
    """Impedance protection flags."""
    return [c for c in cols if FEATURE_TYPES.get(c) == "impedance_flag"]

def cols_for_relays(relays, cols):
    """Support R1:XXX and R1-XXX naming."""
    return [
        c for c in cols
        if any(c.startswith(r + ":") or c.startswith(r + "-") for r in relays)
    ]


# ==================================================
# 2. RAW Relay Health (NO CLAMPING)
# ==================================================

PHYSICAL_TYPES = {
    "voltage_magnitude",
    "current_magnitude",
    "frequency",
    "rocof",
    "impedance_flag",
}

relay_health_raw = {}

for relay in ["R1", "R2", "R3", "R4"]:
    cols = [
        c for c in severity_df.columns
        if (c.startswith(relay + ":") or c.startswith(relay + "-"))
        and FEATURE_TYPES.get(c) in PHYSICAL_TYPES
    ]

    if not cols:
        relay_health_raw[relay] = 0.0
        continue

    row = severity_df[cols]

    relay_health_raw[relay] = np.where(
        (row == 1.0).any(axis=1), 1.0,
        np.where((row == 0.6).any(axis=1), 0.6, 0.0)
    )

relay_health_raw = pd.DataFrame(relay_health_raw, index=df.index)


# ==================================================
# 3. LINE HEALTH (FLAG → RED, CURRENT → AMBER)
# ==================================================

L1_ZONE = cols_for_relays(["R1", "R2"], severity_df.columns)
L2_ZONE = cols_for_relays(["R3", "R4"], severity_df.columns)

L1_FLAGS = flag_cols(L1_ZONE)
L2_FLAGS = flag_cols(L2_ZONE)

L1_I = current_cols(L1_ZONE)
L2_I = current_cols(L2_ZONE)

def line_health_physical(sev_row, flags, currents, other_flags):
    """
    Physical line inference (PRIMARY + BACKUP):
    - RED   : impedance flag OR dual-ended critical current
    - AMBER : single-ended current stress
    - GREEN : otherwise
    """

    # -------------------------
    # PRIMARY: impedance trip
    # -------------------------
    if flags and (sev_row[flags] == 1.0).any():
        return 1.0

    # -------------------------
    # BLOCK if other line tripped
    # -------------------------
    if other_flags and (sev_row[other_flags] == 1.0).any():
        return 0.0

    # -------------------------
    # BACKUP: dual-ended overcurrent
    # -------------------------
    if currents:
        crit_curr = (sev_row[currents] == 1.0).sum()
        warn_curr = (sev_row[currents] >= 0.6).sum()

        # both relays see critical current → fault
        if crit_curr >= 1:
            return 1.0

        # one relay sees stress → amber
        if warn_curr >= 1:
            return 0.6

    return 0.0

line_health = pd.DataFrame(index=df.index)
line_health["L1"] = severity_df.apply(
    lambda r: line_health_physical(r, L1_FLAGS, L1_I, L2_FLAGS),
    axis=1
)

line_health["L2"] = severity_df.apply(
    lambda r: line_health_physical(r, L2_FLAGS, L2_I, L1_FLAGS),
    axis=1
)

# ---- Mutual exclusion (safety net only) ----
both_active = (line_health["L1"] > 0) & (line_health["L2"] > 0)

score_L1 = (severity_df[L1_FLAGS + L1_I] > 0).sum(axis=1) if (L1_FLAGS or L1_I) else 0
score_L2 = (severity_df[L2_FLAGS + L2_I] > 0).sum(axis=1) if (L2_FLAGS or L2_I) else 0

line_health.loc[both_active & (score_L1 >= score_L2), "L2"] = 0.0
line_health.loc[both_active & (score_L2 >  score_L1), "L1"] = 0.0


# ==================================================
# 4. Clamp Relays to Their Line (DISPLAY CONSISTENCY)
# ==================================================

relay_health = relay_health_raw.copy()

RELAY_TO_LINE = {
    "R1": "L1", "R2": "L1",
    "R3": "L2", "R4": "L2",
}

for relay, line in RELAY_TO_LINE.items():
    relay_health[relay] = np.minimum(relay_health[relay], line_health[line])


# ==================================================
# 5. Generator Health (Frequency + ROCOF)
# ==================================================

gen_health = pd.DataFrame(index=df.index)

for gen, cols in {
    "G1": ["R1:F", "R1:DF"],
    "G2": ["R3:F", "R3:DF"],
}.items():

    valid = [c for c in cols if c in severity_df.columns]

    if not valid:
        gen_health[gen] = 0.0
        continue

    row = severity_df[valid]

    gen_health[gen] = np.where(
        (row == 1.0).any(axis=1), 1.0,
        np.where((row == 0.6).any(axis=1), 0.6, 0.0)
    )


# ==================================================
# 6. Assemble FINAL Element Health Table
# ==================================================

element_health_final = pd.concat(
    [gen_health, relay_health, line_health],
    axis=1
)

for x in ["B1", "B2", "B3", "BR1", "BR2", "BR3", "BR4"]:
    element_health_final[x] = 0.0

print("=== FINAL Element Physical Health (FIXED) ===")
display(element_health_final.head())

=== FINAL Element Physical Health (FIXED) ===


,G1,G2,R1,R2,R3,R4,L1,L2,B1,B2,B3,BR1,BR2,BR3,BR4
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.6,0.6,0.0,0.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.6,0.6,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.6,0.6,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
import numpy as np
import pandas as pd

# ==================================================
# CONFIG
# ==================================================

CURRENT_STRESS_THRESHOLD = 3   # << interpretable knob

RELAYS = ["R1", "R2", "R3", "R4"]
LINES = {"L1": ["R1","R2"], "L2": ["R3","R4"]}

# ==================================================
# HELPERS
# ==================================================

def cols_for(relays, cols):
    return [c for c in cols if any(c.startswith(r + ":") or c.startswith(r + "-") for r in relays)]

def current_cols(cols):
    return [c for c in cols if FEATURE_TYPES.get(c) == "current_magnitude"]

def flag_cols(cols):
    return [c for c in cols if FEATURE_TYPES.get(c) == "impedance_flag"]

# ==================================================
# RELAY HEALTH (independent, interpretable)
# ==================================================

relay_health = {}

for r in RELAYS:
    r_cols = cols_for([r], severity_df.columns)
    flags = flag_cols(r_cols)
    currents = current_cols(r_cols)

    def relay_state(row):
        if flags and (row[flags] == 1.0).any():
            return 1.0
        if currents and (row[currents] >= 0.6).any():
            return 0.6
        return 0.0

    relay_health[r] = severity_df.apply(relay_state, axis=1)

relay_health = pd.DataFrame(relay_health, index=df.index)

# ==================================================
# LINE HEALTH (MERGED: protection + localized stress)
# ==================================================

line_health = {}

for line, rels in LINES.items():
    zone = cols_for(rels, severity_df.columns)
    flags = flag_cols(zone)
    currents = current_cols(zone)

    # other line (for blocking / comparison)
    other_line = [l for l in LINES if l != line][0]
    other_zone = cols_for(LINES[other_line], severity_df.columns)
    other_flags = flag_cols(other_zone)
    other_currents = current_cols(other_zone)

    def line_state(row):
        # ----------------------------------
        # 1) TRUE protection decision
        # ----------------------------------
        if flags and (row[flags] == 1.0).any():
            return 1.0

        # ----------------------------------
        # 2) BLOCK if other line is faulted
        # ----------------------------------
        if other_flags and (row[other_flags] == 1.0).any():
            return 0.0

        # ----------------------------------
        # 3) Localized current stress (soft)
        # ----------------------------------
        own_stress = (row[currents] >= 0.6).sum() if currents else 0
        other_stress = (row[other_currents] >= 0.6).sum() if other_currents else 0

        if (
            own_stress >= CURRENT_STRESS_THRESHOLD
            and own_stress > other_stress
        ):
            return 0.6

        return 0.0

    line_health[line] = severity_df.apply(line_state, axis=1)

line_health = pd.DataFrame(line_health, index=df.index)

# ==================================================
# FIX COLOURING: Clamp relays by their line
# ==================================================

RELAY_TO_LINE = {
    "R1": "L1", "R2": "L1",
    "R3": "L2", "R4": "L2",
}

for relay, line in RELAY_TO_LINE.items():
    relay_health[relay] = np.minimum(
        relay_health[relay],
        line_health[line]
    )

# ==================================================
# GENERATOR HEALTH
# ==================================================

gen_health = {}

for g, cols in {"G1":["R1:F","R1:DF"], "G2":["R3:F","R3:DF"]}.items():
    valid = [c for c in cols if c in severity_df.columns]
    if not valid:
        gen_health[g] = 0.0
        continue

    gen_health[g] = np.where(
        (severity_df[valid] == 1.0).any(axis=1), 1.0,
        np.where((severity_df[valid] == 0.6).any(axis=1), 0.6, 0.0)
    )

gen_health = pd.DataFrame(gen_health, index=df.index)

# ==================================================
# FINAL PHYSICAL LAYER
# ==================================================

element_health_final = pd.concat(
    [gen_health, relay_health, line_health],
    axis=1
)

# Non-physical elements
for x in ["B1","B2","B3","BR1","BR2","BR3","BR4"]:
    element_health_final[x] = 0.0

order = ["G1","G2","R1","R2","R3","R4","L1","L2","B1","B2","B3","BR1","BR2","BR3","BR4"]
element_health_final = element_health_final[order]

print("=== FINAL Element Physical Health (MY WAY) ===")
display(element_health_final.head())

=== FINAL Element Physical Health (MY WAY) ===


,G1,G2,R1,R2,R3,R4,L1,L2,B1,B2,B3,BR1,BR2,BR3,BR4
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.6,0.6,0.0,0.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# ==========================================
# DIAGNOSTIC: Current severity distribution
# ==========================================

for line, currents in {
    "L1": L1_I,
    "L2": L2_I
}.items():
    if not currents:
        continue

    print(f"\n=== {line} CURRENT SEVERITY ===")
    print(
        severity_df[currents]
        .apply(lambda r: (r >= 0.6).sum(), axis=1)
        .value_counts()
        .sort_index()
        .head(10)
    )

In [ ]:
# --------------------------------------------------
# TEST 1: Normal operation sanity
# --------------------------------------------------

normal_idx = df[df["marker"] == 41].index

print("=== NORMAL OPERATION TEST ===")
print(f"Normal samples: {len(normal_idx)}")

print("\nElement-level RED count (should be 0):")
print((element_health_final.loc[normal_idx] == 1.0).sum())

print("\nElement-level AMBER count (allowed, should be small):")
print((element_health_final.loc[normal_idx] == 0.6).sum())

In [ ]:
# --------------------------------------------------
# TEST 2: Line activation by scenario
# --------------------------------------------------

SCENARIO_LINE_MAP = {
    1: "L1", 2: "L1", 3: "L1",
    4: "L2", 5: "L2", 6: "L2",
    7: "L1", 8: "L1", 9: "L1",
    10: "L2", 11: "L2", 12: "L2",
}

for marker, line in SCENARIO_LINE_MAP.items():
    idx = df[df["marker"] == marker].index
    if len(idx) == 0:
        continue

    active = (element_health_final.loc[idx, line] > 0).mean()
    other = "L2" if line == "L1" else "L1"
    inactive = (element_health_final.loc[idx, other] > 0).mean()

    print(f"Marker {marker}:")
    print(f"  {line} active ratio    : {active:.2%}")
    print(f"  {other} active ratio   : {inactive:.2%}")

In [ ]:
# --------------------------------------------------
# TEST 3: Relay–Line consistency
# --------------------------------------------------

violations = 0

RELAY_TO_LINE = {"R1": "L1", "R2": "L1", "R3": "L2", "R4": "L2"}

for relay, line in RELAY_TO_LINE.items():
    bad = (element_health_final[relay] == 1.0) & (element_health_final[line] == 0.0)
    violations += bad.sum()

print("=== RELAY–LINE CONSISTENCY ===")
print(f"Violations (should be 0): {violations}")

In [ ]:
# --------------------------------------------------
# TEST 4: Protection flag effectiveness
# --------------------------------------------------

flag_cols = [c for c in severity_df.columns if FEATURE_TYPES.get(c) == "impedance_flag"]

print("Flag columns:", flag_cols)

for col in flag_cols:
    triggered = (df[col] == 1).sum()
    if triggered == 0:
        continue

    # Which line does this flag belong to?
    if col.startswith(("R1", "R2")):
        line = "L1"
    else:
        line = "L2"

    ratio = (element_health_final.loc[df[col] == 1, line] == 1.0).mean()

    print(f"{col}:")
    print(f"  Flag activations      : {triggered}")
    print(f"  Line RED ratio        : {ratio:.2%}")

In [ ]:
def inspect_sample(idx):
    print("=" * 60)
    print(f"Sample index: {idx}")
    print(f"Marker: {df.loc[idx, 'marker']}")
    print("=" * 60)

    # -----------------------------
    # Impedance flags (PHYSICAL)
    # -----------------------------
    print("\n--- Impedance flags (severity) ---")
    found = False
    for c in severity_df.columns:
        if FEATURE_TYPES.get(c) == "impedance_flag" and severity_df.loc[idx, c] == 1.0:
            print(f"  {c} = 1")
            found = True
    if not found:
        print("  (none)")

    # -----------------------------
    # Line health
    # -----------------------------
    print("\n--- Line health ---")
    display(
        element_health_final.loc[idx, ["L1", "L2"]]
        .to_frame("health")
    )

    # -----------------------------
    # Relay health
    # -----------------------------
    print("\n--- Relay health ---")
    display(
        element_health_final.loc[idx, ["R1", "R2", "R3", "R4"]]
        .to_frame("health")
    )

In [ ]:
inspect_sample(df[df["marker"] == 1].index[5])
inspect_sample(df[df["marker"] == 2].index[5])
inspect_sample(df[df["marker"] == 3].index[5])
inspect_sample(df[df["marker"] == 4].index[5])
inspect_sample(df[df["marker"] == 5].index[5])
inspect_sample(df[df["marker"] == 6].index[5])
inspect_sample(df[df["marker"] == 7].index[5])
inspect_sample(df[df["marker"] == 8].index[5])

In [ ]:
# pick a sample where L1 is actually active
fault_idx = (
    element_health_final
    .query("L2 > 0")
    .index[3]
)

inspect_sample(fault_idx)

In [ ]:
def test_sample(idx, df, element_health_final):
    print("=" * 60)
    print(f"Sample index: {idx}")
    print(f"Marker: {df.loc[idx, 'marker']}")
    print("=" * 60)

    row = element_health_final.loc[idx]

    for elem, val in row.items():
        if val == 1.0:
            state = "RED"
        elif val == 0.6:
            state = "AMBER"
        else:
            state = "GREEN"
        print(f"{elem:>4} : {state}")

In [ ]:
# Normal
idx = df[df["marker"] == 41].index[3]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 1].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 2].index[0]
test_sample(idx, df, element_health_final)

# L1 severe fault (marker 3)
idx = df[df["marker"] == 3].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 4].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 5].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 6].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 7].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 8].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 9].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 10].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 11].index[10]
test_sample(idx, df, element_health_final)

# L1 severe fault (marker 3)
idx = df[df["marker"] == 12].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 13].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 14].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 15].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 16].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 17].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 18].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 19].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 20].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 21].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 22].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 23].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 24].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 25].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 26].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 27].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 28].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 29].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 30].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 31].index[0]
test_sample(idx, df, element_health_final)
# Check if any RED in normal
normal_idx = df[df["marker"] == 41].index
print("\nRED elements in normal (should be 0):",
      (element_health_final.loc[normal_idx].max(axis=1) == 1.0).sum())

In [349]:
# --------------------------------------------------
# Aggregate feature severities to element-level score
# --------------------------------------------------

# --------------------------------------------------
# Element health computation (ownership + weights)
# --------------------------------------------------

def compute_element_health(severity_df, element_features, element_weights, feature_types):
    """
    Correct element health:
      health_E(t) = Σ_{type} weight(type) * max_severity_over_owned_features_of_that_type(t)

    severity_df: DataFrame [time x features], values in [0,1]
    element_features: dict {element -> [owned feature names]}
    element_weights: dict {element_type -> {feature_type: weight}}
    feature_types: dict {feature -> feature_type}
    """

    element_health = pd.DataFrame(index=severity_df.index)

    for element, owned in element_features.items():

        # ---- determine element type
        if element.startswith("BR"):
            etype = "breaker"
        elif element.startswith("R"):
            etype = "relay"
        elif element.startswith("L"):
            etype = "line"
        elif element.startswith("G"):
            etype = "generator"
        else:
            raise ValueError(f"Unknown element: {element}")

        weights = element_weights[etype]

        # start at 0 for all time
        score = pd.Series(0.0, index=severity_df.index)

        # apply each weight ONCE per type
        for ftype, w in weights.items():
            cols = [f for f in owned if feature_types.get(f) == ftype]
            if not cols:
                continue

            # max severity across owned features of this type
            type_max = severity_df[cols].max(axis=1)
            score += w * type_max

        element_health[element] = score

    return element_health



# ---- Run it on a SMALL slice first (fast sanity check)
df_small = severity_df.sample(n=500, random_state=42)   # <-- adjust size if needed

element_health = compute_element_health(
    df_small,
    ELEMENT_FEATURES,
    ELEMENT_WEIGHTS,
    FEATURE_TYPES
)

print("Element health shape:", element_health.shape)
element_health.head(50)
element_health_with_marker = element_health.copy()
element_health_with_marker["marker"] = df.loc[element_health.index, "marker"]

element_health_with_marker.head(50)

Element health shape: (500, 12)


,R1,R2,R3,R4,BR1,BR2,BR3,BR4,L1,L2,G1,G2,marker
38854,0.00,0.15,0.15,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.00,0.00,24
74030,0.60,0.69,0.40,0.00,0.30,0.30,0.00,0.00,0.75,0.45,0.40,0.20,36
60133,0.00,0.24,0.24,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.00,0.12,24
36602,0.27,0.33,0.33,0.27,0.18,0.18,0.18,0.18,0.36,0.36,0.24,0.24,41
64691,0.27,0.33,0.33,0.27,0.18,0.18,0.18,0.18,0.36,0.36,0.24,0.12,28
701,0.09,0.15,0.24,0.09,0.00,0.00,0.00,0.00,0.03,0.03,0.00,0.00,37
43587,0.18,0.33,0.33,0.18,0.18,0.18,0.18,0.18,0.36,0.36,0.24,0.24,26
55511,0.00,0.15,0.15,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.12,0.12,19
11306,0.00,0.24,0.24,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.12,0.12,36
19663,0.09,0.15,0.15,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.12,0.12,10


In [74]:
# --------------------------------------------------
# Compute element health time series
# --------------------------------------------------

ELEMENTS = {
    "R1": "relay", "R2": "relay", "R3": "relay", "R4": "relay",
    "BR1": "breaker", "BR2": "breaker", "BR3": "breaker", "BR4": "breaker",
    "L1": "line", "L2": "line",
    "G1": "generator", "G2": "generator"
}

# Randomly sample 200 timesteps
severity_small = severity_df.sample(n=50, random_state=42)
element_health_small[elem] = severity_small.apply(
    lambda row: aggregate_element_severity(
        severity_row=row,
        element=elem,          # 👈 ADD THIS
        element_type=etype,
        feature_types=FEATURE_TYPES
    ),
    axis=1
)
element_health_small.sort_index()
# --------------------------------------------------
# Combine element health with scenario marker
# --------------------------------------------------

element_health_with_marker = element_health_small.copy()
element_health_with_marker["marker"] = df.loc[element_health.index, "marker"]

element_health_with_marker.head(50)
    

,R1,R2,R3,R4,BR1,BR2,BR3,BR4,L1,L2,G1,G2,marker
38854,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,24
74030,0.60,0.60,0.70,0.30,0.3,0.3,0.3,0.3,0.75,0.85,0.4,0.4,36
60133,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,24
36602,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,41
64691,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.60,0.60,0.2,0.2,28
701,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,37
43587,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.60,0.60,0.2,0.2,26
55511,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,19
11306,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,36
19663,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,10


In [453]:
print("\n" + "="*80)
print("TEST 2 — FEATURE TYPE ABNORMALITY CHECK")
print("="*80)

def test_feature_type(feature_type, multiplier=2.0):
    features = bounds_df[bounds_df["type"] == feature_type].index
    if len(features) == 0:
        print(f"[{feature_type}] — no features found")
        return

    f = features[0]
    row = bounds_df.loc[f]

    sample = bounds_df["median"].copy()

    # Inject abnormal value
    if feature_type in ["current_magnitude", "voltage_magnitude"]:
        sample[f] = row["p99"] * multiplier
    elif feature_type in ["phase_angle", "impedance_angle", "rocof"]:
        sample[f] = row["median"] + 4 * max(row["p95"] - row["median"], 1e-6)
    elif feature_type == "impedance":
        sample[f] = max(row["q25"] - 4 * (row["q75"] - row["q25"]), 0)
    elif feature_type == "frequency":
        sample[f] = row["median"] + 1.0
    elif feature_type == "status":
        sample[f] = row["median"] + 1
    else:
        print(f"[{feature_type}] skipped (non-physical)")
        return

    state = feature_state(sample[f], row)

    print(f"\nFeature: {f}")
    print(f"Type: {feature_type}")
    print(f"Injected value: {sample[f]}")
    print(f"Median: {row['median']}")
    print(f"p99: {row.get('p99', 'N/A')}")
    print(f"Resulting state: {state}")

# Run tests for all physical types
for t in [
    "voltage_magnitude",
    "current_magnitude",
    "phase_angle",
    "impedance",
    "impedance_angle",
    "frequency",
    "rocof",
    "status"
]:
    test_feature_type(t)


TEST 2 — FEATURE TYPE ABNORMALITY CHECK

Feature: R1-PM1:V
Type: voltage_magnitude
Injected value: 266428.567
Median: 131659.7408
p99: 133214.2835
Resulting state: RED

Feature: R1-PM4:I
Type: current_magnitude
Injected value: 1130.8580623999999
Median: 393.86961
p99: 565.4290311999999
Resulting state: RED

Feature: R1-PA1:VH
Type: phase_angle
Injected value: 596.5900104000001
Median: -3.443476
p99: 177.30728984
Resulting state: RED

Feature: R1-PA:Z
Type: impedance
Injected value: 0
Median: 9.598383018
p99: 14.826951748399999
Resulting state: AMBER

Feature: R1-PA:ZH
Type: impedance_angle
Injected value: 0.2558122
Median: 0.036937
p99: 0.09997512
Resulting state: RED

Feature: R1:F
Type: frequency
Injected value: 61.0
Median: 60.0
p99: 60.0015
Resulting state: RED

Feature: R1:DF
Type: rocof
Injected value: 4e-06
Median: 0.0
p99: 0.0
Resulting state: RED

Feature: R1:S
Type: status
Injected value: 1.0
Median: 0.0
p99: 0.0
Resulting state: RED


In [425]:
ELEMENT_MAP = {

    "R1": [c for c in bounds_df.index
           if c.startswith("R1-") and (":Z" in c or ":I" in c or c.endswith(":S"))
           and "Z_inf_flag" not in c] + ["relay1_log"],

    "R2": [c for c in bounds_df.index
           if c.startswith("R2-") and (":Z" in c or ":I" in c or c.endswith(":S"))
           and "Z_inf_flag" not in c] + ["relay2_log"],

    "R3": [c for c in bounds_df.index
           if c.startswith("R3-") and (":Z" in c or ":I" in c or c.endswith(":S"))
           and "Z_inf_flag" not in c] + ["relay3_log"],

    "R4": [c for c in bounds_df.index
           if c.startswith("R4-") and (":Z" in c or ":I" in c or c.endswith(":S"))
           and "Z_inf_flag" not in c] + ["relay4_log"],

    "BR1": ["R1:S"],
    "BR2": ["R2:S"],
    "BR3": ["R3:S"],
    "BR4": ["R4:S"],

    "B1": [c for c in bounds_df.index if c.startswith("R1-") and ":V" in c],
    "B2": [c for c in bounds_df.index if c.startswith("R2-") and ":V" in c],
    "B3": [c for c in bounds_df.index if c.startswith("R4-") and ":V" in c],

    "G1": [c for c in bounds_df.index if c.startswith("R1-")
           and (":V" in c or c.endswith(":F") or c.endswith(":DF"))],

    "G2": [c for c in bounds_df.index if c.startswith("R4-")
           and (":V" in c or c.endswith(":F") or c.endswith(":DF"))],
}

LINE_MAP = {
    "L1": {
        "current_features": [c for c in bounds_df.index
                             if (c.startswith("R1-") or c.startswith("R2-")) and ":I" in c],
        "breaker_status": ["R1:S", "R2:S"]
    },
    "L2": {
        "current_features": [c for c in bounds_df.index
                             if (c.startswith("R3-") or c.startswith("R4-")) and ":I" in c],
        "breaker_status": ["R3:S", "R4:S"]
    }
}

In [442]:
# =========================
# TEST 1: Normal behaviour
# =========================

normal_idx = df[df["marker"] == 41].index[0]
sample = df.loc[normal_idx].reindex(FEATURES)

feature_states = evaluate_feature_states(sample, bounds_df)

from collections import Counter
print("Normal feature state counts:")
print(Counter(feature_states.values()))

Normal feature state counts:
Counter({'GREEN': 111, 'AMBER': 19, 'RED': 2})


In [443]:
# =========================
# TEST 2: Fault current
# =========================

fault_idx = df[df["marker"].isin([1,2,3,4,5,6])].index[0]
sample = df.loc[fault_idx].reindex(FEATURES)

feature_states = evaluate_feature_states(sample, bounds_df)

# Show only current features
for f, state in feature_states.items():
    if FEATURE_TYPES[f] == "current_magnitude":
        print(f"{f:25s} -> {state}")

R1-PM4:I                  -> GREEN
R1-PM5:I                  -> GREEN
R1-PM6:I                  -> GREEN
R1-PM10:I                 -> GREEN
R1-PM11:I                 -> GREEN
R1-PM12:I                 -> GREEN
R2-PM4:I                  -> GREEN
R2-PM5:I                  -> GREEN
R2-PM6:I                  -> GREEN
R2-PM10:I                 -> GREEN
R2-PM11:I                 -> GREEN
R2-PM12:I                 -> GREEN
R3-PM4:I                  -> GREEN
R3-PM5:I                  -> GREEN
R3-PM6:I                  -> GREEN
R3-PM10:I                 -> GREEN
R3-PM11:I                 -> GREEN
R3-PM12:I                 -> GREEN
R4-PM4:I                  -> GREEN
R4-PM5:I                  -> GREEN
R4-PM6:I                  -> GREEN
R4-PM10:I                 -> GREEN
R4-PM11:I                 -> GREEN
R4-PM12:I                 -> GREEN
